# 🧠 LSTM: Deep Learning ETA Prediction

**Purpose:** Train LSTM model on real EMS dataset

**Input:**
- train_real.csv (8,000 samples)
- val_real.csv (1,000 samples)
- test_real.csv (1,000 samples)

**Model:**
- 2 LSTM layers (128, 64 units)
- Dropout & batch normalization
- Early stopping on val MAE

**Target:**
- MAE < 3.9 min (beat RF baseline)
- RMSE < 3.0 min

## STEP 1: Mount Drive & Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print('✅ Google Drive mounted')

# Set paths
DATA_PATH = '/content/drive/MyDrive/NaviRaksha_Output'
DATA_PROCESSED = f'{DATA_PATH}/processed'
MODELS_PATH = f'{DATA_PATH}/models'

os.makedirs(MODELS_PATH, exist_ok=True)
print(f'✅ Paths configured')

## STEP 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f'✅ TensorFlow version: {tf.__version__}')
print(f'✅ GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## STEP 3: Load & Preprocess Data

In [ ]:
# Load datasets
train_df = pd.read_csv(f'{DATA_PROCESSED}/train_real.csv')
val_df = pd.read_csv(f'{DATA_PROCESSED}/val_real.csv')
test_df = pd.read_csv(f'{DATA_PROCESSED}/test_real.csv')

print(f'✅ Train: {train_df.shape}')
print(f'✅ Val: {val_df.shape}')
print(f'✅ Test: {test_df.shape}')

# Feature columns
TARGET = 'eta_minutes'
DROP_COLS = ['trip_id', 'month', 'eta_minutes']
feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

# Prepare data
X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df[TARGET].values.astype(np.float32)
X_val = val_df[feature_cols].values.astype(np.float32)
y_val = val_df[TARGET].values.astype(np.float32)
X_test = test_df[feature_cols].values.astype(np.float32)
y_test = test_df[TARGET].values.astype(np.float32)

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Normalize targets
y_mean = y_train.mean()
y_std = y_train.std()
y_train_norm = (y_train - y_mean) / y_std
y_val_norm = (y_val - y_mean) / y_std
y_test_norm = (y_test - y_mean) / y_std

print(f'\n✅ Features: {X_train_scaled.shape}')
print(f'✅ Targets normalized (mean={y_mean:.3f}, std={y_std:.3f})')

## STEP 4: Build LSTM Model

In [ ]:
# Build LSTM model
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    layers.LSTM(128, activation='relu', return_sequences=True),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    layers.LSTM(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mae',
    metrics=['mae', 'mse']
)

print(model.summary())

## STEP 5: Train Model

In [ ]:
# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Train
print('🚀 Training LSTM...')
history = model.fit(
    X_train_scaled, y_train_norm,
    validation_data=(X_val_scaled, y_val_norm),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print('\n✅ Training complete')

## STEP 6: Evaluate Model

In [ ]:
# Predictions (denormalized)
pred_train_norm = model.predict(X_train_scaled, verbose=0)
pred_val_norm = model.predict(X_val_scaled, verbose=0)
pred_test_norm = model.predict(X_test_scaled, verbose=0)

pred_train = (pred_train_norm * y_std) + y_mean
pred_val = (pred_val_norm * y_std) + y_mean
pred_test = (pred_test_norm * y_std) + y_mean

# Clip to valid range
pred_train = np.clip(pred_train, 3, 15)
pred_val = np.clip(pred_val, 3, 15)
pred_test = np.clip(pred_test, 3, 15)

# Metrics
def calc_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name}: MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}')
    return mae, rmse, r2

print('\n' + '='*70)
print('📊 LSTM PERFORMANCE')
print('='*70)
mae_train, rmse_train, r2_train = calc_metrics(y_train, pred_train, 'Train')
mae_val, rmse_val, r2_val = calc_metrics(y_val, pred_val, 'Val')
mae_test, rmse_test, r2_test = calc_metrics(y_test, pred_test, 'Test')

print(f'\n✅ Test MAE: {mae_test:.4f} min')
print(f'   Target: < 3.9 min')
print(f'   Status: {"✅ ACHIEVED" if mae_test < 3.9 else "⚠️  CLOSE"}')

## STEP 7: Plot Training History

In [ ]:
# Plot history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MAE Loss')
axes[0].set_title('Training History: Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Actual vs Predicted
axes[1].scatter(y_test, pred_test, alpha=0.5, s=20)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual ETA (min)')
axes[1].set_ylabel('Predicted ETA (min)')
axes[1].set_title('Test Set: Actual vs Predicted')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## STEP 8: Save Model

In [ ]:
# Save model
model.save(f'{MODELS_PATH}/lstm_best_real.h5')
print(f'✅ Model saved: lstm_best_real.h5')

# Save scaler info
import json
scaler_info = {
    'mean': y_mean.item(),
    'std': y_std.item()
}
with open(f'{MODELS_PATH}/lstm_scaler.json', 'w') as f:
    json.dump(scaler_info, f)
print(f'✅ Scaler info saved')

## ✅ SUMMARY

In [ ]:
print('\n' + '='*70)
print('✅ LSTM TRAINING COMPLETE')
print('='*70)

print(f'\n📊 Performance:')
print(f'   Train MAE: {mae_train:.4f} min')
print(f'   Val MAE:   {mae_val:.4f} min')
print(f'   Test MAE:  {mae_test:.4f} min')

print(f'\n🎯 Comparison with RF Baseline:')
print(f'   RF Baseline MAE: 4.2+ min')
print(f'   LSTM Test MAE:   {mae_test:.4f} min')
improvement = 4.2 - mae_test
print(f'   Improvement:     {improvement:.4f} min ({(improvement/4.2*100):.1f}%)')

print(f'\n📁 Saved Models:')
print(f'   ✓ models/lstm_best_real.h5')
print(f'   ✓ models/lstm_scaler.json')

print(f'\n🚀 Next: Create GNN model (04_gnn_training.ipynb)')